## Clean Cosing


## Step 1: Load Annex II and inspect it (separator, columns, “Moved or deleted” marker) 
Load Annex II and inspect its structure. Before cleaning all five annexes, I check one file: the separator, column names, and where the “Moved or deleted” marker appears. That way the loop in Step 2 is built on facts rather than guesses.

In [4]:
import pandas as pd                      # pandas for tables

# path to Annex II (notebook runs from notebook/, so go up one level)
path_ii = "../data/raw/cosing/COSING_Annex_II_v2.txt"

# read the file; skip the 4 header lines of the export
# hint: if everything lands in ONE column, the separator is wrong -> try sep="\t" or sep=";"
annex_ii = pd.read_csv(path_ii, skiprows=4, sep=",", encoding="utf-8")   # try "utf-8" first, "latin-1" if it errors

print(annex_ii.shape)                    # expect about 1758 rows
print(annex_ii.columns.tolist())         # list of column names
annex_ii.head(5)                         # first 3 rows

(1758, 11)
['Reference Number', 'Chemical name / INN', 'CAS Number', 'EC Number', 'Regulation', 'Other Directives/Regulations', 'SCCS opinions', 'Chemical/IUPAC Name', 'Identified INGREDIENTS or substances e.g.', 'CMR', 'Update Date']


,Reference Number,Chemical name / INN,CAS Number,EC Number,Regulation,Other Directives/Regulations,SCCS opinions,Chemical/IUPAC Name,Identified INGREDIENTS or substances e.g.,CMR,Update Date
0,1,N-(5-Chlorobenzoxazol-2-yl)acetamide,35783-57-4,-,(EC) 2009/1223,NaN,NaN,NaN,NaN,NaN,17/10/2010
1,2,(2-Acetoxyethyl)trimethylammonium hydroxide (A...,51-84-3,200-128-9,(EC) 2009/1223,NaN,NaN,NaN,NaN,NaN,17/10/2010
2,3,Deanol aceglumate (INN),3342-61-8,222-085-5,(EC) 2009/1223,NaN,NaN,NaN,NaN,NaN,15/10/2010
3,4,Spironolactone (INN),52-01-7,200-133-6,(EC) 2009/1223,NaN,NaN,NaN,SPIRONOLACTONE,NaN,15/10/2010
4,5,"[4-(4-Hydroxy-3-iodophenoxy)-3,5-diiodophenyl]...",51-24-1,200-086-1,(EC) 2009/1223,NaN,NaN,Triiodothyroacetic acid; (4-(4-Hydroxy-3-iodop...,NaN,NaN,17/10/2010


In [3]:
# turn every cell into text, check if it contains "moved or deleted" (case-insensitive)
mask = annex_ii.astype(str).apply(lambda col: col.str.contains("moved or deleted", case=False, na=False))

# how many hits per column? -> tells us WHICH column holds the marker
print(mask.sum()[mask.sum() > 0])

# how many ROWS have the marker anywhere?
print(mask.any(axis=1).sum())

Chemical name / INN    18
dtype: int64
18


**Findings**
- Annex II is comma-separated, 11 columns, 1758 rows. The “Moved or deleted” marker appears only in Chemical name / INN (18 rows). These are dead entries: the substance was moved to another annex or removed. Expected after dropping: 1758 − 18 = 1740 rows. Missing values in EC Number are written as “-”, not NaN, and CAS may be the same. I’ll handle that in the CAS step.

## Step 2:One function load_annex() that reads all five annexes and drops the 24 dead rows
Load all five annexes with one reusable function. It reads a file, drops “Moved or deleted” rows (searching all columns, because column names differ between annexes), and reports rows before/after. Then I compare the column lists to plan how to combine them.

In [5]:
def load_annex(roman):
    """Read one CosIng annex file and drop 'Moved or deleted' rows."""
    path = f"../data/raw/cosing/COSING_Annex_{roman}_v2.txt"     # build the file name from the roman numeral
    df = pd.read_csv(path, skiprows=4, sep=",", encoding="utf-8")

    # same mask as before: True where any cell contains the marker
    mask = df.astype(str).apply(lambda col: col.str.contains("moved or deleted", case=False, na=False))
    dead = mask.any(axis=1)                                    # one True/False per row

    print(f"Annex {roman}: {len(df)} rows, {dead.sum()} moved/deleted -> {len(df) - dead.sum()} kept")
    return df[~dead].copy()                                    # keep the rows that are NOT dead

# load all five into a dictionary: {"II": df, "III": df, ...}
annexes = {r: load_annex(r) for r in ["II", "III", "IV", "V", "VI"]}

Annex II: 1758 rows, 18 moved/deleted -> 1740 kept
Annex III: 381 rows, 2 moved/deleted -> 379 kept
Annex IV: 154 rows, 0 moved/deleted -> 154 kept
Annex V: 58 rows, 3 moved/deleted -> 55 kept
Annex VI: 34 rows, 1 moved/deleted -> 33 kept


In [6]:
# print each annex's column names so we can see what they share
for roman, df in annexes.items():           # hint: loop over key AND value of a dict
    print(roman, df.columns.tolist())
    print()

II ['Reference Number', 'Chemical name / INN', 'CAS Number', 'EC Number', 'Regulation', 'Other Directives/Regulations', 'SCCS opinions', 'Chemical/IUPAC Name', 'Identified INGREDIENTS or substances e.g.', 'CMR', 'Update Date']

III ['Reference Number', 'Chemical name / INN', 'Name of Common Ingredients Glossary', 'CAS Number', 'EC Number', 'Product Type, body parts', 'Maximum concentration in ready for use preparation', 'Other', 'Wording of conditions of use and warnings', 'Regulation', 'Other Directives/Regulations', 'SCCS opinions', 'Chemical/IUPAC Name', 'Identified INGREDIENTS or substances e.g.', 'CMR', 'Update Date']

IV ['Reference Number', 'Chemical name', 'Colour index Number / Name of Common Ingredients Glossary', 'CAS Number', 'EC Number', 'Color', 'Product Type, body parts', 'Maximum concentration in ready for use preparation', 'Other', 'Wording of conditions of use and warnings', 'Regulation', 'Other Directives/Regulations', 'SCCS opinions', 'Chemical/IUPAC Name', 'Identif

**Findings**
- All annexes read with the same settings. 24 moved/deleted rows dropped, 2361 rows kept. The columns are almost the same across annexes, with two differences: (1) Annex IV calls the name column Chemical name instead of Chemical name / INN; (2) Annex II has no Maximum concentration column, because banned substances have no allowed limit. CAS Number and CMR exist in all five.

## Step 3:Standardise the column names and stack all annexes into eu_status (2361 × 14)
Step 3: Standardise column names and stack all annexes into one table eu_status, plus an annex column. I keep every column a later step needs: CAS/EC for joins, INCI names for Sephora matching, max concentration, product type and warnings for the restriction details, CMR and SCCS opinions as the hazard link, and the update date for the EU-vs-California timeline. Columns missing in one annex (e.g. max concentration in Annex II) become NaN.

In [23]:
rename_map = {
    "Reference Number": "ref_no",
    "Chemical name / INN": "name",
    "Chemical name": "name",                                              # Annex IV's name column
    "Name of Common Ingredients Glossary": "inci_name",          # INCI name (III, V, VI) -> Sephora matching
    "Colour index Number / Name of Common Ingredients Glossary": "inci_name",   # Annex IV's version of the same idea
    "Identified INGREDIENTS or substances e.g.": "inci_examples",  # INCI names listed in Annex II
    "CAS Number": "cas",
    "EC Number": "ec",
    "Product Type, body parts": "product_type",
    "Maximum concentration in ready for use preparation": "max_conc",
    "Other": "other_limits",
    "Wording of conditions of use and warnings": "warnings",
    "CMR": "cmr",
    "SCCS opinions": "sccs_opinion",
    "Update Date": "last_uptodate",                                        # pick a short name
}
keep = ["ref_no", "name", "inci_name", "inci_examples", "cas", "ec",
        "product_type", "max_conc", "other_limits", "warnings",
        "cmr", "sccs_opinion", "last_uptodate"]                            # same short name as above

parts = []
for roman, df in annexes.items():
    tmp = df.rename(columns= rename_map )
    tmp = tmp.reindex(columns=keep)                              # missing columns become NaN
    tmp["annex"] = roman
    parts.append(tmp)

eu_status = pd.concat(parts, ignore_index=True)

print(eu_status.shape)                                           # expect (2361, 14)
print(eu_status["annex"].value_counts())
#eu_status.sample(5, random_state=1)

(2361, 14)
annex
II     1740
III     379
IV      154
V        55
VI       33
Name: count, dtype: int64


In [9]:
eu_status.sample(10)

,ref_no,name,inci_name,inci_examples,cas,ec,product_type,max_conc,other_limits,warnings,cmr,sccs_opinion,last_uptodate,annex
1256,1281,4-[(4-Nitrophenyl)azo]aniline (Disperse Orange...,NaN,DISPERSE ORANGE 3,730-40-5,211-984-8,NaN,NaN,NaN,NaN,NaN,Opinion concerning Use of Permanent Hair Dyes ...,09/09/2020,II
204,205,Methocarbamol (INN),NaN,NaN,532-03-6,208-524-3,NaN,NaN,NaN,NaN,NaN,NaN,15/10/2010,II
256,257,Noradrenaline and its salts,NaN,NaN,51-41-2,200-096-6,NaN,NaN,NaN,NaN,NaN,NaN,15/10/2010,II
1767,23,(a) Alkali sulphides \r\n(b) Alkaline earth su...,-,"BARIUM SULFIDE,CALCIUM SULFIDE,LITHIUM SULFIDE...",-,-,(a) Depilatories \r\n(b) Depilatories,(a) 2% (as sulphur)\r\n(b) 6% (as sulphur),pH,(a) (b) \r\nKeep out of reach of children \r\n...,NaN,NaN,17/10/2010,III
1850,112,Pinus palustris oil and extract,-,"PINUS PALUSTRIS LEAF EXTRACT,PINUS PALUSTRIS T...",97435-14-8 -- 8002-09-3,306-895-7,NaN,NaN,Peroxide value less than 10 mmoles/L (15 ),(15 ) This limit applies to the substance and ...,NaN,An Initial List of Perfumery Materials which m...,17/05/2013,III
526,532,"Gases (petroleum), thermal cracking distn., if...",NaN,NaN,68478-05-7,270-789-6,NaN,NaN,NaN,NaN,"Carcinogenic Cat. 1B(),Mutagenic Cat. 1B()",Opinion concerning Chemical Ingredients in Cos...,15/10/2010,II
1139,1160,"4,4'-Oxydianiline (p-aminophenyl ether) and it...",NaN,NaN,101-80-4,202-977-0,NaN,NaN,NaN,NaN,"Carcinogenic Cat. 1B(),Mutagenic Cat. 1B(),Rep...",Opinion concerning Request for Confirmation of...,15/10/2010,II
1400,1428,"Nickel diformate [1];\r\nFormic acid, nickel s...",NaN,NaN,3349-06-2 [1] 15843-02-4 [2] 68134-59-8 [3],222-101-0 [1] 239-946-6 [2] 268-755-0 [3],NaN,NaN,NaN,NaN,"Carcinogenic Cat. 1B(),Reprotoxic Cat. 1B(),Mu...",NaN,30/07/2020,II
643,651,"1,2-Dibromoethane",NaN,NaN,106-93-4,203-444-5,NaN,NaN,NaN,NaN,Carcinogenic Cat. 1B(),Opinion concerning Chemical Ingredients in Cos...,15/10/2010,II
1175,1200,"Chlorotoluron; (3-(3-Chloro-p-tolyl)-1,1-dimet...",NaN,NaN,15545-48-9,239-592-2,NaN,NaN,NaN,NaN,"Carcinogenic Cat. 2(),Reprotoxic Cat. 2()",Opinion concerning Request for Confirmation of...,15/10/2010,II


**Findings**
- eu_status has 2361 rows × 14 columns. Observations: (1) several CAS numbers can share one cell, separated by “ - “, and since CAS numbers contain dashes, a simple split won’t work; (2) some text cells contain line breaks (\r\n) and need stripping; (3) ref_no restarts per annex, so the unique key is (annex, ref_no). The CMR column gives the hazard class, e.g. “Carcinogenic Cat. 1B”.

In [10]:
cas_txt = eu_status["cas"].astype(str)

# how many cells are "-" or empty (no CAS)?
print((cas_txt.str.strip().isin(["-", "", "nan"])).sum())

# cells that contain something other than digits and dashes -> probably several CAS or notes
odd = eu_status[cas_txt.str.contains(r"[^0-9\-]", regex=True)]
print(len(odd))
print(odd["cas"].head(15).tolist())                          # show 15 examples

89
268
['132-60-5 / 5949-18-8', '8063-06-7 / 22260-42-0', '51-28-5 / 329-71-5 / 573-56-8 / 25550-58-7', '97-77-8 / 137-26-8', '103-16-2 / 622-62-8', '58-89-9 / -', '118-08-1 / 6592-85-4', '156-74-1 / 541-22-0 / 1420-40-2 / 3198-38-7', '91-59-8 / 134-32-7', '14402-89-2 / 13755-38-9', '2858-66-4 / 4396-01-4', '555-77-1 / 817-09-4 / 6138-32-5', '50-14-6 / 67-97-0', '58-73-1 / 147-24-0 / 88637-37-0', '298-81-7 / 484-20-8 / 3902-71-4']


## Step 4:Find every CAS number by pattern and split to one row per CAS: eu_cas (3,074 rows, 2,894 unique CAS)
Some cells list several CAS numbers (separated by “ / “ or “ - “), and CAS numbers contain dashes themselves. So I don’t split on a separator. I search each cell for the CAS pattern and give every CAS its own row. eu_status stays one row per entry; eu_cas has one row per CAS, which I need for joining to California.

In [24]:
# 1. Clean text columns: convert everything to text FIRST (integers would otherwise become NaN), then strip
for col in eu_status.select_dtypes(include=["object", "str"]).columns:
    eu_status[col] = eu_status[col].astype("string").str.strip()     # "string" type keeps missing values as <NA>

# 2. Find every CAS number in each cell
cas_pattern = r"\d{2,7}-\d{2}-\d"                                # CAS format: 2-7 digits, dash, 2 digits, dash, 1 digit
eu_status["cas_list"] = (eu_status["cas"]
                         .astype(str)                            # make sure every cell is text (NaN becomes "nan")
                         .str.findall(cas_pattern))              # returns a LIST of all CAS found in the cell ([] if none)

# 3. One row per CAS number
eu_cas = eu_status.explode("cas_list")                           # a cell with 3 CAS becomes 3 rows; an empty list becomes NaN
eu_cas = eu_cas.rename(columns={"cas_list": "cas_single"})       # clearer name: one single CAS per row

# 4. Remove duplicate CAS within the same annex, but ONLY for rows that have a CAS
has_cas = eu_cas["cas_single"].notna()                           # True where a CAS exists
eu_cas = pd.concat(
    [eu_cas[has_cas].drop_duplicates(subset=["annex", "cas_single"]),  # CAS rows: drop repeats of (annex, CAS)
     eu_cas[~has_cas]],                                                # no-CAS rows (groups): keep all, they are different entries
    ignore_index=True)                                                 # fresh row numbers 0..n

# 5. Checks (expected: 2361 entries -> 3074 rows, 2894 unique CAS)
print("entries:", eu_status.shape[0], "-> rows after split:", eu_cas.shape[0])
print("rows without CAS per annex:")
print(eu_cas[eu_cas["cas_single"].isna()].groupby("annex").size())
print("unique CAS:", eu_cas["cas_single"].nunique())

entries: 2361 -> rows after split: 3074
rows without CAS per annex:
annex
II     69
III    17
IV      5
dtype: int64
unique CAS: 2894


In [25]:
print(eu_status["ref_no"].isna().sum(), "missing ref_no")                    # expect 0
print(eu_status.groupby("annex")["ref_no"].agg(rows="size", distinct="nunique"))

0 missing ref_no
       rows  distinct
annex                
II     1740      1738
III     379       370
IV      154       154
V        55        55
VI       33        33


**Findings:**
- After splitting, eu_cas has 3,074 rows and 2,894 unique CAS numbers. 91 entries have no CAS (69 in Annex II, 17 in III, 5 in IV). They are mostly substance groups and can’t be matched by CAS. This is a limitation of the matching: group bans (e.g. “lead and its compounds”) are not detected.

## Step 5: Test California against Annex II and III: 38 banned, 20 restricted
This is the key test of the project. I match the California CAS numbers against the EU annexes: Annex II (banned) and Annex III (restricted). Result: how many of the chemicals reported in California cosmetics are banned or restricted in the EU.

In [14]:
# California chemicals: one row per CAS (drop rows without a valid CAS)
ca = pd.read_csv("../data/clean/california.csv")
ca_chem = (ca.dropna(subset=["CasNumber"])[["CasNumber", "ChemicalName"]]
             .drop_duplicates()
             .reset_index(drop=True))
print("California chemicals with CAS:", len(ca_chem))

# sets of EU CAS numbers per annex
cas_II  = set(eu_cas.loc[eu_cas["annex"] == "II",  "cas_single"].dropna())
cas_III = set(eu_cas.loc[eu_cas["annex"] == "III", "cas_single"].dropna())

# flags: is this California chemical in the EU ban / restriction list?
ca_chem["eu_banned"]     = ca_chem["CasNumber"].isin(cas_II)
ca_chem["eu_restricted"] = ca_chem["CasNumber"].isin(cas_III)

print(ca_chem[["eu_banned", "eu_restricted"]].sum())

# which ones are banned?
ca_chem[ca_chem["eu_banned"]].sort_values("ChemicalName")

California chemicals with CAS: 107
eu_banned        38
eu_restricted    20
dtype: int64


,CasNumber,ChemicalName,eu_banned,eu_restricted
40,123-91-1,"1,4-Dioxane",True,False
95,3296-90-0,"2,2-Bis(bromomethyl)-1,3-propanediol",True,False
39,75-07-0,Acetaldehyde,True,False
17,79-06-1,Acrylamide,True,False
68,302-79-4,All-trans retinoic acid,True,False
97,56-55-3,Benz[a]anthracene,True,False
45,71-43-2,Benzene,True,False
96,50-32-8,Benzo[a]pyrene,True,False
54,119-61-9,Benzophenone,True,False
46,100-44-7,Benzyl chloride,True,False


**Findings**
- 107 California chemicals have a valid CAS. 38 rows (37 unique CAS) are in EU Annex II (banned) and 20 in Annex III (restricted). Many hits are impurities (1,4-dioxane, benzene, nitrosamines) that the EU only tolerates as unavoidable traces. A CAS can be in more than one annex, so for the final status Annex II takes priority over III.

## Step 6: Save the EU tables, add the flags to all California reports, count products and companies, compare removal rates
Save the cleaned EU tables for SQL and Tableau. Then carry the EU flags over to every California report, to count how many products, companies and reports are affected, and compare removal rates.

In [26]:
eu_status.drop(columns="cas_list").to_csv("../data/clean/eu_status.csv", index=False)
eu_cas.to_csv("../data/clean/eu_cas.csv", index=False)

In [17]:
# 1. Save the EU tables
eu_status.drop(columns="cas_list").to_csv("../data/clean/eu_status.csv", index=False)  # one row per annex entry (list column removed)
eu_cas.to_csv("../data/clean/eu_cas.csv", index=False)                                  # one row per CAS (no list column here)

# 2. Overlap check: are some CAS both banned and restricted?
print("banned AND restricted:", (ca_chem["eu_banned"] & ca_chem["eu_restricted"]).sum())

# 3. Add the EU flags to every California report row (left join on CAS)
flags = ca_chem[["CasNumber", "eu_banned", "eu_restricted"]].drop_duplicates("CasNumber")  # one row per CAS
ca = ca.merge(flags, on="CasNumber", how="left")            # rows without a CAS get NaN
ca["eu_banned"]     = ca["eu_banned"].fillna(False)         # unknown CAS counts as "not flagged"
ca["eu_restricted"] = ca["eu_restricted"].fillna(False)

# 4. How many reports, products and companies are affected?
print("reports with EU-banned chemical:", ca["eu_banned"].sum(), "of", len(ca))
print("products affected:", ca.loc[ca["eu_banned"], "CDPHId"].nunique(), "of", ca["CDPHId"].nunique())
print("companies affected:", ca.loc[ca["eu_banned"], "CompanyId"].nunique(), "of", ca["CompanyId"].nunique())

# 5. Removal rate: banned vs not banned (mean of True/False = share of True)
print(ca.groupby("eu_banned")["is_removed"].mean())

# 6. Same without titanium dioxide, because TiO2 is 81.8% of all rows
print(ca[~ca["is_tio2"]].groupby("eu_banned")["is_removed"].mean())

banned AND restricted: 0
reports with EU-banned chemical: 845 of 114381
products affected: 455 of 36972
companies affected: 86 of 635
eu_banned
False    0.025049
True     0.166864
Name: is_removed, dtype: float64
eu_banned
False    0.059580
True     0.166864
Name: is_removed, dtype: float64


In [18]:
# 1. Row count must stay 114,381 (a faulty merge would add rows)
print(ca.shape)                                   # expect (114381, 28): 26 old columns + eu_banned + eu_restricted

# 2. No duplicated flag columns (_x / _y would appear if the merge ran twice)
print([c for c in ca.columns if c.endswith(("_x", "_y"))])    # expect []

# 3. The flag columns exist exactly once
print(ca.columns.tolist().count("eu_banned"), ca.columns.tolist().count("eu_restricted"))   # expect 1 1

(114381, 28)
[]
1 1


**Finfings:**
845 of 114,381 reports (0.7%) contain a chemical that is banned in the EU, affecting 455 of 36,972 products and 86 of 635 companies. These chemicals were removed more often (16.7%) than the rest (6.0%, without TiO2). This is only a first descriptive result. Whether the difference is statistically significant is tested on Thursday with a chi-square test (H1).

## Step 7: Save california_eu.csv and inspect the inventory (30,080 rows)
Save California with the EU flags as its own file, so the original stays unchanged. Then load the CosIng ingredient inventory (INCI names and functions) and inspect it before cleaning. It is the bridge to the Sephora ingredient lists.

In [20]:
# 1. Save California with the EU flags (new file, keeps the Day 1 file unchanged)
ca.to_csv("../data/clean/california_eu.csv", index=False)

# 2. Load the CosIng inventory (path from the earlier file check)
inv = pd.read_csv("../data/raw/cosing/COSING_Ingredients_Fragrance_Inventory_v2.csv",
                  encoding="utf-8")             # try "latin-1" if this gives an error

# 3. Inspect: size, columns, first rows, missing values
print(inv.shape)                                # number of rows and columns
print(inv.columns.tolist())                     # column names
print(inv.isna().sum())                         # missing values per column
inv.head(5)                                     # first 5 rows

(30080, 10)
['COSING Ref No', 'INCI name', 'INN name', 'Ph. Eur. Name', 'CAS No', 'EC No', 'Chem/IUPAC Name / Description', 'Restriction', 'Function', 'Update Date']
COSING Ref No                        0
INCI name                            0
INN name                         29561
Ph. Eur. Name                    29897
CAS No                               0
EC No                                0
Chem/IUPAC Name / Description      471
Restriction                      28112
Function                           216
Update Date                          0
dtype: int64


,COSING Ref No,INCI name,INN name,Ph. Eur. Name,CAS No,EC No,Chem/IUPAC Name / Description,Restriction,Function,Update Date
0,94753,DISODIUM TETRAMETHYLHEXADECENYLCYSTEINE FORM...,NaN,NaN,"2040469-40-5, 2422121-34-2",,Disodium Tetramethylhexadecenylcysteine Formyl...,NaN,SKIN PROTECTING,16/06/2020
1,99268,(ANGELICA ACUTILOBA/PAEONIA LACTIFLORA) ROOT/...,NaN,NaN,,,(Angelica Acutiloba/Paeonia Lactiflora) Root/C...,NaN,SKIN CONDITIONING - EMOLLIENT,19/03/2021
2,99270,(ARCTIUM LAPPA/RAPHANUS SATIVUS/MALVA VERTICI...,NaN,NaN,,,(Arctium Lappa/Raphanus Sativus/Malva Verticil...,NaN,SKIN CONDITIONING - EMOLLIENT,19/03/2021
3,94896,(LIQUIDAMBAR STYRACIFLUA/TRIBULUS TERRESTRIS)...,NaN,NaN,,,(Liquidambar Styraciflua/Tribulus Terrestris)...,NaN,SKIN CONDITIONING,01/12/2017
4,95645,ACRYLATES/VA/VINYL NEODECANOATE COPOLYMER,NaN,NaN,99728-55-9,,Acrylates/VA/Vinyl Neodecanoate Copolymer is ...,NaN,PLASTICISER,14/02/2018


## Step 8: Clean the inventory: column names, UPPER INCI, “-” becomes missing
Clean the CosIng inventory for matching with Sephora. INCI names are written in capitals, so I standardise them (strip + UPPER) to match despite spelling differences. The Function column can list several functions in one cell, so I split it to one row per function (needed for the “what does it do” analysis). inv_names keeps one row per ingredient for joining.

In [21]:
# 1. Short snake_case column names (same style as the annexes)
inv = inv.rename(columns={
    "COSING Ref No": "ref_no",
    "INCI name": "inci_name",
    "INN name": "inn_name",
    "Ph. Eur. Name": "ph_eur_name",
    "CAS No": "cas",
    "EC No": "ec",
    "Chem/IUPAC Name / Description": "description",
    "Restriction": "restriction",
    "Function": "function",
    "Update Date": "last_update"})

# 2. Remove spaces/line breaks in all text columns
for col in inv.select_dtypes(include=["object", "str"]).columns:
    inv[col] = inv[col].str.strip()

# 3. Standardise INCI names: UPPER case, so "Aqua" and "AQUA" match later
inv["inci_name"] = inv["inci_name"].str.upper()

# 4. Placeholders "-" mean "no value": turn them into real missing values
inv[["cas", "ec"]] = inv[["cas", "ec"]].replace("-", pd.NA)

# 5. Check: are INCI names unique? And which separator do the functions use?
print("rows:", len(inv), "| unique INCI names:", inv["inci_name"].nunique())
print(inv["function"].dropna().sample(8, random_state=1).tolist())    # look at 8 examples of the Function column

rows: 30080 | unique INCI names: 30079
['BINDING', 'ANTISTATIC, HAIR CONDITIONING, PRESERVATIVE', 'PERFUMING', 'SURFACTANT - CLEANSING, SURFACTANT - EMULSIFYING', 'CHELATING', 'HAIR CONDITIONING, SKIN CONDITIONING', 'ABRASIVE, BULKING, ORAL CARE, SKIN CONDITIONING', 'SKIN CONDITIONING']


## Step 9: Split functions to one row per function and save the inventory tables
- One INCI name occurs twice, so I check it before using INCI as a join key. Then I split the comma-separated functions into one row per ingredient and function (inv_functions) and keep one row per ingredient (inv_names). Both are saved for SQL and the Sephora matching.

In [22]:
# 1. Show the INCI name that appears twice
dup = inv[inv["inci_name"].duplicated(keep=False)]               # keep=False shows BOTH rows of the duplicate
print(dup[["ref_no", "inci_name", "cas", "function"]])

# 2. One row per ingredient: keep the first occurrence of each INCI name
inv_names = (inv.drop(columns="function")                        # function goes into its own table
               .drop_duplicates(subset="inci_name")              # INCI becomes a unique key
               .reset_index(drop=True))

# 3. One row per ingredient AND function
inv_functions = inv[["inci_name", "function"]].dropna(subset=["function"]).copy()   # drop the 216 without function
inv_functions["function"] = inv_functions["function"].str.split(",")                # "A, B" becomes ["A", "B"]
inv_functions = inv_functions.explode("function")                                   # one row per function
inv_functions["function"] = inv_functions["function"].str.strip()                   # remove spaces left by the split
inv_functions = inv_functions.drop_duplicates().reset_index(drop=True)              # no repeated pairs

# 4. Checks
print("ingredients:", inv_names.shape[0], "| ingredient-function rows:", inv_functions.shape[0])
print(inv_functions["function"].value_counts().head(10))        # 10 most common functions

# 5. Save for SQL / Sephora matching
inv_names.to_csv("../data/clean/inventory_names.csv", index=False)
inv_functions.to_csv("../data/clean/inventory_functions.csv", index=False)

       ref_no                                          inci_name  cas  \
14597  100082  LACTOBACILLUS/PANAX GINSENG ROOT FERMENT FILTRATE        
14598   82328  LACTOBACILLUS/PANAX GINSENG ROOT FERMENT FILTRATE  NaN   

                                function  
14597  SKIN CONDITIONING - MISCELLANEOUS  
14598                  SKIN CONDITIONING  
ingredients: 30079 | ingredient-function rows: 53278
function
SKIN CONDITIONING                13035
SKIN CONDITIONING - EMOLLIENT     3339
HAIR CONDITIONING                 3332
SURFACTANT - CLEANSING            3073
SURFACTANT - EMULSIFYING          2746
PERFUMING                         2711
ANTIOXIDANT                       2559
SKIN PROTECTING                   2356
HUMECTANT                         2089
FRAGRANCE                         1405
Name: count, dtype: int64


## Summary: CosIng cleaning and EU matching

| Item | Result |
|---|---|
| Annexes loaded (II, III, IV, V, VI) | 2,385 rows, 24 "moved or deleted" rows dropped, 2,361 kept |
| `eu_status` | 2,361 entries x 14 columns (key: annex + ref_no) |
| `eu_cas` | 3,074 rows, 2,894 unique CAS; 91 entries without CAS (groups) |
| California chemicals with CAS | 107 |
| Banned in the EU (Annex II) | 38 rows (37 unique CAS) |
| Restricted in the EU (Annex III) | 20 |
| Reports with a banned chemical | 845 of 114,381 (0.7%) |
| Products / companies affected | 455 of 36,972 / 86 of 635 |
| Removal rate, banned vs other (without TiO2) | 16.7% vs 6.0% |
| Inventory | 30,079 unique INCI names, 53,278 ingredient-function rows |

Limitations: group entries without a CAS (e.g. "lead and its compounds") cannot be matched by CAS; many banned hits are impurities that the EU tolerates as unavoidable traces.
Files saved: eu_status.csv, eu_cas.csv, california_eu.csv, inventory_names.csv, inventory_functions.csv